In [1]:
# %pip install sentence-transformers --break-system-packages

In [2]:
# %pip install rank-bm25 --break-system-packages

In [3]:
# from sentence_transformers import CrossEncoder

# reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
# print("리랭커 로드 완료")

In [4]:
# ===== 1: 설정 =====
import json
from pathlib import Path
from pymilvus import MilvusClient, DataType
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
CHUNKS_DIR = PROJECT_DIR / "data" / "chunks"

CHUNK_FILES = {
    "강아지": CHUNKS_DIR / "petmint_강아지.json",
    "고양이": CHUNKS_DIR / "petmint_고양이.json",
}

MILVUS_URI = "http://localhost:19530"
EMBEDDING_MODEL_NAME = "nlpai-lab/KURE-v1"

COLLECTION_NAMES = {
    "강아지": "petmint_dog",
    "고양이": "petmint_cat",
}

def load_chunks(species: str) -> list[dict]:
    with open(CHUNK_FILES[species], "r", encoding="utf-8") as f:
        return json.load(f)

chunks_dog = load_chunks("강아지")
chunks_cat = load_chunks("고양이")

print("강아지 청크 수:", len(chunks_dog))
print("고양이 청크 수:", len(chunks_cat))

print("\n임베딩 모델 로드 중...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("임베딩 차원:", embedding_model.get_sentence_embedding_dimension())

/home/kopo17/llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


강아지 청크 수: 928
고양이 청크 수: 850

임베딩 모델 로드 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11511.57it/s]

임베딩 차원: 1024


/tmp/ipykernel_381264/1037128164.py:35: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("임베딩 차원:", embedding_model.get_sentence_embedding_dimension())


In [5]:
# ===== 2: Milvus 컬렉션 생성 (강아지/고양이 분리) =====

client = MilvusClient(uri=MILVUS_URI)

def create_collection(collection_name: str):
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    schema = client.create_schema(auto_id=True, enable_dynamic_field=True)
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("vector", DataType.FLOAT_VECTOR, dim=1024)
    schema.add_field("content", DataType.VARCHAR, max_length=12000)
    schema.add_field("species", DataType.VARCHAR, max_length=10)
    schema.add_field("section", DataType.VARCHAR, max_length=64)
    schema.add_field("section_name", DataType.VARCHAR, max_length=256, nullable=True)
    schema.add_field("type", DataType.VARCHAR, max_length=5, nullable=True)
    schema.add_field("article_num", DataType.INT64, nullable=True)
    schema.add_field("article_title", DataType.VARCHAR, max_length=256, nullable=True)
    schema.add_field("coverage_type", DataType.VARCHAR, max_length=10, nullable=True)
    schema.add_field("paragraph_num", DataType.INT64, nullable=True)
    schema.add_field("chunk_index", DataType.INT64)
    schema.add_field("chunk_total", DataType.INT64)
    schema.add_field("printed_page_start", DataType.INT64)
    schema.add_field("printed_page_end", DataType.INT64)
    schema.add_field("citations_raw", DataType.VARCHAR, max_length=4000, nullable=True)
    # 분할된 뒷조각의 맥락(01번 context_prefix) — search_engine 헤더가 dense/BM25 양쪽에서 같도록 Milvus에도 둔다
    schema.add_field("context_prefix", DataType.VARCHAR, max_length=1024, nullable=True)

    index_params = client.prepare_index_params()
    index_params.add_index(field_name="vector", index_type="AUTOINDEX", metric_type="COSINE")

    client.create_collection(collection_name=collection_name, schema=schema, index_params=index_params)
    print(f"{collection_name} 컬렉션 생성 완료")

for species, name in COLLECTION_NAMES.items():
    create_collection(name)

petmint_dog 컬렉션 생성 완료


petmint_cat 컬렉션 생성 완료


In [6]:
# ===== 3: 임베딩 생성 + Milvus 삽입 =====

import sys
sys.path.insert(0, str(PROJECT_DIR))
from data.search_engine import build_embedding_text   # 검색 시점(BM25·리랭커)과 같은 텍스트로 색인 — 복사본을 두지 않는다

def citation_to_string(cit: dict) -> str:
    """citation dict를 사람이 읽기 좋은 문자열로 요약 (citations_raw 저장용)"""
    parts = []
    if cit.get("law_name"):
        parts.append(cit["law_name"])
    elif cit.get("doc_name"):
        parts.append(cit["doc_name"])
    if cit.get("article_num") is not None:
        parts.append(f"제{cit['article_num']}조")
    if cit.get("paragraph_num") is not None and cit.get("paragraph_type"):
        parts.append(f"제{cit['paragraph_num']}{cit['paragraph_type']}")
    if cit.get("sub_ho") is not None:
        parts.append(f"제{cit['sub_ho']}호")
    return " ".join(parts) if parts else "?"

def insert_chunks(chunks: list[dict], species: str, batch_size: int = 32):
    collection_name = COLLECTION_NAMES[species]
    total = len(chunks)

    for start in range(0, total, batch_size):
        batch = chunks[start:start + batch_size]
        embedding_texts = [build_embedding_text(c) for c in batch]
        vectors = embedding_model.encode(embedding_texts, normalize_embeddings=True)

        data = []
        for c, v in zip(batch, vectors):
            citations_raw = ", ".join(citation_to_string(cit) for cit in c.get("citations", [])) or None
            data.append({
                "vector": v.tolist(),
                "content": c["content"],
                "species": c["species"],
                "section": c["section"],
                "section_name": c.get("section_name"),
                "type": c.get("type"),
                "article_num": c.get("article_num"),
                "article_title": c.get("article_title"),
                "coverage_type": c.get("coverage_type"),
                "paragraph_num": c.get("paragraph_num"),
                "chunk_index": c["chunk_index"],
                "chunk_total": c["chunk_total"],
                "printed_page_start": c["printed_page_range"][0],
                "printed_page_end": c["printed_page_range"][1],
                "citations_raw": citations_raw,
                "context_prefix": c.get("context_prefix"),
            })

        client.insert(collection_name=collection_name, data=data)
        print(f"{species}: {min(start+batch_size, total)}/{total} 삽입 완료")

insert_chunks(chunks_dog, "강아지")
insert_chunks(chunks_cat, "고양이")

print("\n전체 삽입 완료")

강아지: 32/928 삽입 완료


강아지: 64/928 삽입 완료


강아지: 96/928 삽입 완료


강아지: 128/928 삽입 완료


강아지: 160/928 삽입 완료


...

고양이: 736/850 삽입 완료


고양이: 768/850 삽입 완료


고양이: 800/850 삽입 완료


고양이: 832/850 삽입 완료


고양이: 850/850 삽입 완료

전체 삽입 완료


In [7]:
# ===== 4: 검색 테스트 =====

def search(query: str, species: str, top_k: int = 3):
    collection_name = COLLECTION_NAMES[species]
    query_vector = embedding_model.encode([query], normalize_embeddings=True)[0].tolist()

    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=top_k,
        output_fields=["content", "section", "section_name", "type", "article_num", "article_title", "coverage_type"],
    )

    for hit in results[0]:
        entity = hit["entity"]
        print(f"\n[score={hit['distance']:.3f}] {entity['section']} / {entity.get('section_name')} / type={entity.get('type')} / 제{entity.get('article_num')}조({entity.get('article_title')}) {entity.get('coverage_type') or ''}")
        print(entity["content"][:150])

search("고급형 수술 한도가 얼마인가요?", "강아지", top_k=10)


[score=0.482] 특별약관 / 펫퍼민트 반려견 입원의료비Ⅲ보장 특별약관 / type=Ⅲ / 제1조(보험금의 지급사유) 고급형
 【보험금 지급금액[자기부담금 3만원 예시]】
 ①입원 중 수술을 하지 않은 경우(보상비율 70% 가입,
     MRI,CT 및 내시경처치를 받지 않은 날)
 ·피보험자가 부담한 치료비 33만원
 ·보험금 지급금액
     = [(33만원 - 3만원)×70%, 30만

[score=0.474] 특별약관 / 펫퍼민트 반려견 입원의료비Ⅲ보장 특별약관 / type=Ⅲ / 제1조(보험금의 지급사유) 고급형
수술과 MRI,CT 및 내시경처치를 동일한 날에 시행한 경
우 수술한 날의 지급한도 내에서 보험금이 지급됩니다.

[score=0.469] 특별약관 / 펫퍼민트 반려견 입원의료비Ⅱ보장 특별약관 / type=Ⅱ / 제1조(보험금의 지급사유) 고급형
회사가 보상하는 비용은 각 항목별 피보험자가 부담한
치료비에서 보험증권에 기재된 자기부담금을 각각 차감한
후, 보험증권에 기재된 보상비율(70%)을 곱한 금액을 아래
에서 정한 금액을 한도로 보상합니다.

      
      항목                    

[score=0.468] 특별약관 / 펫퍼민트 반려견 입원의료비보장 특별약관 / type=Ⅰ / 제3조(수술의 정의와 장소) 
제1항의「수술」은 자택 등에서의 치료가 곤란하여 동물
병원에서 행한 것에 한합니다.

[score=0.466] 특별약관 / 펫퍼민트 반려견 입원의료비보장 특별약관 / type=Ⅰ / 제1조(보험금의 지급사유) 고급형
회사가 보상하는 비용은 각 항목별 피보험자가 부담한
치료비에서 보험증권에 기재된 자기부담금을 각각 차감한
후, 보험증권에 기재된 보상비율(70%)을 곱한 금액을 아래
에서 정한 금액을 한도로 보상합니다.

      
      항목                    

[score=0.466] 특별약관 / 펫퍼민트 반려견 입원의료비Ⅱ보장 특별약관 / type=Ⅱ / 제3조(수술의 정의